In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run /Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/config_loader

In [0]:
def load_config_from_widget():
    try:
        environment = dbutils.widgets.get("environment")
    except:
        environment = "dev"
    
    storage_account = "databricksete123"
    project_name = "orders-analytics-pipeline"
    
    if environment == 'dev':
        config = {
            "environment": "dev",
            "catalog": "databricks_catalog_new",
            "schemas": {"bronze": "dev_bronze", "silver": "dev_silver", "gold": "dev_gold"},
            "storage": {
                "account": storage_account,
                "landing": {
                    "orders": f"abfss://source@{storage_account}.dfs.core.windows.net/orders/",
                    "customers": f"abfss://source@{storage_account}.dfs.core.windows.net/customers/",
                    "products": f"abfss://source@{storage_account}.dfs.core.windows.net/products/",
                    "region": f"abfss://source@{storage_account}.dfs.core.windows.net/region/"
                },
                "external": {
                    "bronze": f"abfss://claude-bronze@{storage_account}.dfs.core.windows.net/dev/{project_name}",
                    "silver": f"abfss://claude-silver@{storage_account}.dfs.core.windows.net/dev/{project_name}",
                    "gold": f"abfss://claude-gold@{storage_account}.dfs.core.windows.net/dev/{project_name}"
                }
            }
        }
    elif environment == 'preprod':
        config = {
            "environment": "preprod",
            "catalog": "databricks_catalog_new",
            "schemas": {"bronze": "preprod_bronze", "silver": "preprod_silver", "gold": "preprod_gold"},
            "storage": {
                "account": storage_account,
                "landing": {
                    "orders": f"abfss://source@{storage_account}.dfs.core.windows.net/orders/",
                    "customers": f"abfss://source@{storage_account}.dfs.core.windows.net/customers/",
                    "products": f"abfss://source@{storage_account}.dfs.core.windows.net/products/",
                    "region": f"abfss://source@{storage_account}.dfs.core.windows.net/region/"
                },
                "external": {
                    "bronze": f"abfss://claude-bronze@{storage_account}.dfs.core.windows.net/preprod/{project_name}",
                    "silver": f"abfss://claude-silver@{storage_account}.dfs.core.windows.net/preprod/{project_name}",
                    "gold": f"abfss://claude-gold@{storage_account}.dfs.core.windows.net/preprod/{project_name}"
                }
            }
        }
    else:  # prod
        config = {
            "environment": "prod",
            "catalog": "databricks_catalog_new",
            "schemas": {"bronze": "prod_bronze", "silver": "prod_silver", "gold": "prod_gold"},
            "storage": {
                "account": storage_account,
                "landing": {
                    "orders": f"abfss://source@{storage_account}.dfs.core.windows.net/orders/",
                    "customers": f"abfss://source@{storage_account}.dfs.core.windows.net/customers/",
                    "products": f"abfss://source@{storage_account}.dfs.core.windows.net/products/",
                    "region": f"abfss://source@{storage_account}.dfs.core.windows.net/region/"
                },
                "external": {
                    "bronze": f"abfss://claude-bronze@{storage_account}.dfs.core.windows.net/prod/{project_name}",
                    "silver": f"abfss://claude-silver@{storage_account}.dfs.core.windows.net/prod/{project_name}",
                    "gold": f"abfss://claude-gold@{storage_account}.dfs.core.windows.net/prod/{project_name}"
                }
            }
        }
    
    return config

config = load_config_from_widget()
print(f"🚀 Bronze Orders Pipeline - {config['environment'].upper()} Environment")
print("Status: Running bronze ingestion...")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, when

orders_path = config['storage']['landing']['orders']
print(f"📂 Reading from: {orders_path}")

df_orders = (spark.read.format("parquet").load(orders_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("environment", lit(config['environment']))
    .withColumn("order_priority",
        when(col("total_amount") < 100, "Low")
        .when(col("total_amount") < 500, "Medium")
        .when(col("total_amount") < 1000, "High")
        .otherwise("Urgent")
    )
    .withColumn("customer_segment",
        when(col("total_amount") >= 1000, "VIP")
        .when(col("total_amount") >= 500, "Premium")
        .when(col("total_amount") >= 100, "Standard")
        .otherwise("Budget")
    )
)

bronze_schema = config['schemas']['bronze']
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{bronze_schema}")

external_path = f"{config['storage']['external']['bronze']}/orders_raw"
bronze_table = f"{config['catalog']}.{bronze_schema}.orders_raw"

print(f"📂 Writing to: {external_path}")

df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .option("path", external_path) \
    .saveAsTable(bronze_table)

print(f"✅ {bronze_table}: {spark.table(bronze_table).count()} records")
print(f"✅ Delta logs created at: {external_path}/_delta_log/")